<a href="https://colab.research.google.com/github/p-tech/wbs-dm-2026/blob/main/web_scrape/football_crazy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =================================================================
# FOOTBALL CRAZY SCRAPING EXERCISE
# Copy and paste this entire script into a single cell in Colab.
# =================================================================

# STEP 1: Setup
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

print("--- Step 1: Libraries Imported ---")

# STEP 2: Fetch the Webpage
URL = "https://footballcrazy.shop/product-category/keyrings-keys/"
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
}

print(f"--- Step 2: Fetching URL: {URL} ---")
response = requests.get(URL, headers=headers)

if response.status_code == 200:
    print("Success! Page received.")
    soup = BeautifulSoup(response.content, "html.parser")

    # STEP 3: Extraction Logic
    print("--- Step 3: Extracting Data ---")
    products_list = []

    # We look for the <li> tags that contain products
    items = soup.find_all("li", class_="product")

    for item in items:
        # Extract Title
        name_tag = item.find("h3", class_="product-title")
        name = name_tag.get_text(strip=True) if name_tag else "N/A"

        # Extract Price
        price_tag = item.find("span", class_="woocommerce-Price-amount")
        price = price_tag.get_text(strip=True) if price_tag else "N/A"

        # Extract Link: Based on your provided snippet, it's in a.product-images
        link_tag = item.find("a", class_="product-images")

        # Fallback to standard link if not found
        if not link_tag:
            link_tag = item.find("a", href=True)

        raw_link = link_tag["href"] if link_tag else "N/A"
        # Ensure the link is a full URL (joins relative paths with the site base)


        products_list.append({
            "Name": name,
            "Price": price,
            "URL": raw_link
        })

    # STEP 4: Results & Export
    print(f"--- Step 4: Processing {len(products_list)} items ---")
    df = pd.DataFrame(products_list)

    # Display the result in the notebook
    from google.colab import data_table
    display(data_table.DataTable(df, include_index=False))

    # Save to CSV
    df.to_csv("football_keyrings.csv", index=False)
    print("\n✅ DONE! Download 'football_keyrings.csv' from the file icon on the left.")

else:
    print(f"Error: Could not access site. Status code: {response.status_code}")

# Pagination

In [ ]:
# =================================================================
# FOOTBALL CRAZY SCRAPING EXERCISE
# Copy and paste this entire script into a single cell in Colab.
# =================================================================

# STEP 1: Setup
import requests
from bs4 import BeautifulSoup
import pandas as pd
from urllib.parse import urljoin
import time

print("--- Step 1: Libraries Imported ---")

# STEP 2: Fetch the Webpage (with Pagination)
base_url = "https://footballcrazy.shop/product-category/keyrings-keys/"
current_url = base_url
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
}

products_list = []
page_number = 1
MAX_PAGES = 5  # Set the limit for pagination

print(f"--- Step 2 & 3: Starting Extraction (Limit: {MAX_PAGES} pages) ---")

while current_url and page_number <= MAX_PAGES:
    print(f"Scraping Page {page_number}: {current_url}...")
    response = requests.get(current_url, headers=headers)

    if response.status_code != 200:
        print(f"Error: Could not access site. Status code: {response.status_code}")
        break

    soup = BeautifulSoup(response.content, "html.parser")

    # Extraction Logic
    # We look for the main wrappers for each product
    items = soup.find_all("div", class_="fusion-product-wrapper")

    # If the above fails, fallback to the standard WooCommerce li items
    if not items:
        items = soup.find_all("li", class_="product")

    for item in items:
        # Extract Title: Looking for h2 or h3 based on site structure
        name_tag = item.find(["h2", "h3"], class_=["woocommerce-loop-product__title", "product-title"])
        name = name_tag.get_text(strip=True) if name_tag else "N/A"

        # Extract Price
        price_tag = item.find("span", class_="woocommerce-Price-amount")
        price = price_tag.get_text(strip=True) if price_tag else "N/A"

        # Extract Link: Based on your provided snippet, it's in a.product-images
        link_tag = item.find("a", class_="product-images")
        # Fallback to standard link if not found
        if not link_tag:
            link_tag = item.find("a", href=True)

        raw_link = link_tag["href"] if link_tag else "N/A"
        # Ensure the link is a full URL (joins relative paths with the site base)
        full_link = urljoin(base_url, raw_link) if raw_link != "N/A" else "N/A"

        products_list.append({
            "Name": name,
            "Price": price,
            "URL": full_link
        })

    # PAGINATION LOGIC: Look for the "Next" page button
    # Common WooCommerce pagination classes: 'next', 'next page-numbers'
    next_button = soup.find("a", class_="next")

    if next_button and next_button.get("href"):
        current_url = urljoin(base_url, next_button["href"])
        page_number += 1
        time.sleep(1) # Polite delay to avoid hammering the server
    else:
        current_url = None # Exit loop if no next button is found

# STEP 4: Results & Export
print(f"--- Step 4: Processing {len(products_list)} total items from {page_number} pages ---")
df = pd.DataFrame(products_list)

if not df.empty:
    # Display the result in the notebook
    from google.colab import data_table
    display(data_table.DataTable(df, include_index=False))

    # Save to CSV
    df.to_csv("football_keyrings_all_pages.csv", index=False)
    print("\n✅ DONE! Download 'football_keyrings_all_pages.csv' from the file icon on the left.")
else:
    print("⚠️ No products found. The site structure might have changed.")